# E004 surface-free deployment candidate
Pure standard-library inference. Internet and external model datasets are not required.
The notebook locates competition data dynamically and writes `/kaggle/working/submission.csv`.


In [ ]:
"""Pure-standard-library E004 inference runtime.

This module intentionally has no project-internal imports. The exact source can be
embedded into a Kaggle notebook and executed with competition data plus a frozen
JSON model artifact only.
"""
from __future__ import annotations

import csv
import hashlib
import json
import math
import statistics
from pathlib import Path
from typing import Any, Mapping, Sequence

MISSING_VALUES = {"", "nan", "NaN", "NA", "N/A", "null", "None"}
REQUIRED_HORIZONTAL = ("MD", "X", "Y", "Z", "GR", "TVT_input")


class DeploymentDataError(ValueError):
    """Raised when inference inputs violate the frozen deployment contract."""


def optional_float(raw: str | None) -> float | None:
    if raw is None or raw.strip() in MISSING_VALUES:
        return None
    try:
        value = float(raw)
    except ValueError:
        return None
    return value if math.isfinite(value) else None


def mean(values: Sequence[float]) -> float:
    return sum(values) / len(values)


def std(values: Sequence[float]) -> float:
    if not values:
        return 0.0
    center = mean(values)
    return math.sqrt(sum((value - center) ** 2 for value in values) / len(values))


def linear_slope(xs: Sequence[float], ys: Sequence[float]) -> float:
    if len(xs) != len(ys) or len(xs) < 2:
        return 0.0
    center_x = mean(xs)
    center_y = mean(ys)
    denominator = sum((value - center_x) ** 2 for value in xs)
    if denominator <= 0.0:
        return 0.0
    return sum((x - center_x) * (y - center_y) for x, y in zip(xs, ys)) / denominator


def paired(xs: Sequence[float], values: Sequence[float | None]) -> tuple[list[float], list[float]]:
    clean_x: list[float] = []
    clean_y: list[float] = []
    for x, value in zip(xs, values):
        if value is not None and math.isfinite(value):
            clean_x.append(float(x))
            clean_y.append(float(value))
    return clean_x, clean_y


def series_summary(xs: Sequence[float], values: Sequence[float | None]) -> dict[str, float | None]:
    clean_x, clean_y = paired(xs, values)
    if not clean_y:
        return {"mean": None, "std": None, "range": None, "delta": None, "slope": None}
    return {
        "mean": mean(clean_y),
        "std": std(clean_y),
        "range": max(clean_y) - min(clean_y),
        "delta": clean_y[-1] - clean_y[0],
        "slope": linear_slope(clean_x, clean_y),
    }


def add_summary(features: dict[str, float | None], prefix: str, xs: Sequence[float], values: Sequence[float | None]) -> None:
    for name, value in series_summary(xs, values).items():
        features[f"{prefix}_{name}"] = value


def missing_fraction(values: Sequence[float | None]) -> float:
    return sum(value is None for value in values) / max(1, len(values))


def longest_missing_run(values: Sequence[float | None]) -> int:
    longest = current = 0
    for value in values:
        if value is None:
            current += 1
            longest = max(longest, current)
        else:
            current = 0
    return longest


def weighted_line(xs: Sequence[float], ys: Sequence[float], weights: Sequence[float]) -> tuple[float, float]:
    total = sum(weights)
    if len(xs) < 2 or total <= 0.0:
        return (ys[-1] if ys else 0.0), 0.0
    center_x = sum(w * x for w, x in zip(weights, xs)) / total
    center_y = sum(w * y for w, y in zip(weights, ys)) / total
    denominator = sum(w * (x - center_x) ** 2 for w, x in zip(weights, xs))
    slope = 0.0 if denominator <= 0.0 else sum(
        w * (x - center_x) * (y - center_y)
        for w, x, y in zip(weights, xs, ys)
    ) / denominator
    return center_y - slope * center_x, slope


def huber_line(
    xs: Sequence[float],
    ys: Sequence[float],
    *,
    window_rows: int,
    huber_k: float = 1.5,
    iterations: int = 6,
) -> tuple[float, float, float]:
    count = min(len(xs), len(ys), max(2, int(window_rows)))
    x = [float(value) for value in xs[-count:]]
    y = [float(value) for value in ys[-count:]]
    weights = [1.0] * count
    intercept, slope = weighted_line(x, y, weights)
    scale = 0.0
    for _ in range(max(1, int(iterations))):
        residuals = [target - (intercept + slope * value) for value, target in zip(x, y)]
        median_residual = statistics.median(residuals)
        absolute = [abs(value - median_residual) for value in residuals]
        scale = 1.4826 * statistics.median(absolute) if absolute else 0.0
        if scale <= 1e-12:
            break
        cutoff = float(huber_k) * scale
        weights = [1.0 if abs(value) <= cutoff else cutoff / abs(value) for value in residuals]
        intercept, slope = weighted_line(x, y, weights)
    return intercept, slope, scale


def visible_backtests(md: Sequence[float], z: Sequence[float], tvt: Sequence[float], fractions: Sequence[float]) -> dict[str, float | None]:
    features: dict[str, float | None] = {}
    count = len(tvt)
    for fraction in fractions:
        label = str(float(fraction)).replace(".", "p")
        cut = min(count - 2, max(2, int(round(count * float(fraction)))))
        if cut < 2 or count - cut < 2:
            for suffix in ("mean", "rmse", "trend", "toe", "u_slope_delta"):
                features[f"backtest_{label}_{suffix}"] = None
            continue
        baseline = tvt[cut - 1]
        residuals = [value - baseline for value in tvt[cut:]]
        centered = [index / max(1, len(residuals) - 1) - 0.5 for index in range(len(residuals))]
        features[f"backtest_{label}_mean"] = mean(residuals)
        features[f"backtest_{label}_rmse"] = math.sqrt(sum(value * value for value in residuals) / len(residuals))
        features[f"backtest_{label}_trend"] = linear_slope(centered, residuals)
        features[f"backtest_{label}_toe"] = residuals[-1]
        before_u = [target + vertical for target, vertical in zip(tvt[:cut], z[:cut])]
        after_u = [target + vertical for target, vertical in zip(tvt[cut:], z[cut:])]
        features[f"backtest_{label}_u_slope_delta"] = linear_slope(md[cut:], after_u) - linear_slope(md[:cut], before_u)
    return features


def read_horizontal(path: Path, *, require_truth: bool = False) -> dict[str, Any]:
    columns: dict[str, list[float | None]] = {}
    fieldnames: list[str] = []
    with path.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        fieldnames = list(reader.fieldnames or [])
        required = set(REQUIRED_HORIZONTAL) | ({"TVT"} if require_truth else set())
        missing = required - set(fieldnames)
        if missing:
            raise DeploymentDataError(f"{path}: missing required columns {sorted(missing)}")
        columns = {name: [] for name in fieldnames}
        for row in reader:
            for name in fieldnames:
                columns[name].append(optional_float(row.get(name)))
    row_count = len(columns.get("MD", []))
    if row_count < 4:
        raise DeploymentDataError(f"{path}: requires at least four rows")
    for name in ("MD", "X", "Y", "Z"):
        if any(value is None for value in columns[name]):
            raise DeploymentDataError(f"{path}: missing or non-finite {name}")
    md = [float(value) for value in columns["MD"] if value is not None]
    if any(second <= first for first, second in zip(md, md[1:])):
        raise DeploymentDataError(f"{path}: MD must be strictly increasing")
    known_rows = 0
    hidden_seen = False
    for value in columns["TVT_input"]:
        visible = value is not None
        if visible and hidden_seen:
            raise DeploymentDataError(f"{path}: TVT_input visibility is not a contiguous prefix")
        if visible:
            known_rows += 1
        else:
            hidden_seen = True
    if known_rows < 3:
        raise DeploymentDataError(f"{path}: requires at least three visible TVT_input rows")
    if known_rows >= row_count:
        raise DeploymentDataError(f"{path}: requires a non-empty hidden suffix")
    if require_truth:
        if any(value is None for value in columns["TVT"]):
            raise DeploymentDataError(f"{path}: training TVT must be complete")
        max_delta = max(
            abs(float(columns["TVT"][index]) - float(columns["TVT_input"][index]))
            for index in range(known_rows)
        )
        if max_delta > 1e-9:
            raise DeploymentDataError(f"{path}: TVT_input differs from TVT on visible rows")
    return {"columns": columns, "fieldnames": fieldnames, "row_count": row_count, "known_rows": known_rows}


def read_typewell(path: Path) -> dict[str, float | None]:
    tvt: list[float | None] = []
    gr: list[float | None] = []
    with path.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        missing = {"TVT", "GR"} - set(reader.fieldnames or [])
        if missing:
            raise DeploymentDataError(f"{path}: missing typewell columns {sorted(missing)}")
        for row in reader:
            tvt.append(optional_float(row.get("TVT")))
            gr.append(optional_float(row.get("GR")))
    if not tvt:
        raise DeploymentDataError(f"{path}: empty typewell")
    clean_tvt = [value for value in tvt if value is not None]
    clean_gr = [value for value in gr if value is not None]
    if not clean_tvt:
        raise DeploymentDataError(f"{path}: typewell TVT is entirely missing")
    index_x, paired_gr = paired([float(index) for index in range(len(gr))], gr)
    return {
        "typewell_rows": float(len(tvt)),
        "typewell_tvt_span": max(clean_tvt) - min(clean_tvt),
        "typewell_gr_mean": mean(clean_gr) if clean_gr else None,
        "typewell_gr_std": std(clean_gr) if clean_gr else None,
        "typewell_gr_range": max(clean_gr) - min(clean_gr) if clean_gr else None,
        "typewell_gr_index_slope": linear_slope(index_x, paired_gr) if paired_gr else None,
        "typewell_gr_missing_fraction": missing_fraction(gr),
    }


def extract_well_features(
    horizontal_path: Path,
    typewell_path: Path,
    *,
    visible_slope_windows: Sequence[int],
    visible_backtest_fractions: Sequence[float],
    require_truth: bool = False,
) -> dict[str, Any]:
    horizontal = read_horizontal(horizontal_path, require_truth=require_truth)
    columns = horizontal["columns"]
    total_rows = int(horizontal["row_count"])
    known_rows = int(horizontal["known_rows"])
    hidden_rows = total_rows - known_rows
    md = [float(value) for value in columns["MD"] if value is not None]
    x = [float(value) for value in columns["X"] if value is not None]
    y = [float(value) for value in columns["Y"] if value is not None]
    z = [float(value) for value in columns["Z"] if value is not None]
    visible_tvt = [float(value) for value in columns["TVT_input"][:known_rows] if value is not None]
    known_md, hidden_md = md[:known_rows], md[known_rows:]
    known_x, hidden_x = x[:known_rows], x[known_rows:]
    known_y, hidden_y = y[:known_rows], y[known_rows:]
    known_z, hidden_z = z[:known_rows], z[known_rows:]
    visible_u = [target + vertical for target, vertical in zip(visible_tvt, known_z)]
    features: dict[str, float | None] = {
        "total_rows": float(total_rows),
        "known_rows": float(known_rows),
        "hidden_rows": float(hidden_rows),
        "hidden_fraction": hidden_rows / total_rows,
        "md_known_span": known_md[-1] - known_md[0],
        "md_hidden_span": hidden_md[-1] - known_md[-1],
        "md_total_span": md[-1] - md[0],
        "last_visible_tvt": visible_tvt[-1],
        "last_visible_z": known_z[-1],
        "last_visible_u": visible_u[-1],
        "last_visible_x": known_x[-1],
        "last_visible_y": known_y[-1],
        "x_total_delta": x[-1] - x[0],
        "y_total_delta": y[-1] - y[0],
        "z_total_delta": z[-1] - z[0],
        "x_hidden_delta": hidden_x[-1] - known_x[-1],
        "y_hidden_delta": hidden_y[-1] - known_y[-1],
        "z_hidden_delta": hidden_z[-1] - known_z[-1],
        "horizontal_total_distance": math.hypot(x[-1] - x[0], y[-1] - y[0]),
        "horizontal_hidden_distance": math.hypot(hidden_x[-1] - known_x[-1], hidden_y[-1] - known_y[-1]),
        "spatial_x_mid": 0.5 * (x[0] + x[-1]),
        "spatial_y_mid": 0.5 * (y[0] + y[-1]),
    }
    angle = math.atan2(hidden_y[-1] - known_y[-1], hidden_x[-1] - known_x[-1])
    features["hidden_azimuth_sin"] = math.sin(angle)
    features["hidden_azimuth_cos"] = math.cos(angle)
    add_summary(features, "visible_tvt", known_md, visible_tvt)
    add_summary(features, "visible_z", known_md, known_z)
    add_summary(features, "visible_u", known_md, visible_u)
    add_summary(features, "hidden_z", hidden_md, hidden_z)
    add_summary(features, "visible_gr", known_md, columns["GR"][:known_rows])
    add_summary(features, "hidden_gr", hidden_md, columns["GR"][known_rows:])
    add_summary(features, "whole_gr", md, columns["GR"])
    features["visible_gr_missing_fraction"] = missing_fraction(columns["GR"][:known_rows])
    features["hidden_gr_missing_fraction"] = missing_fraction(columns["GR"][known_rows:])
    features["whole_gr_missing_fraction"] = missing_fraction(columns["GR"])
    features["hidden_gr_longest_missing_run"] = float(longest_missing_run(columns["GR"][known_rows:]))
    full_u_slope = linear_slope(known_md, visible_u)
    for window in [int(value) for value in visible_slope_windows]:
        count = min(window, known_rows)
        if count < 2:
            features[f"visible_u_slope_w{window}"] = None
            features[f"visible_u_scale_w{window}"] = None
            features[f"visible_u_slope_delta_w{window}"] = None
            features[f"visible_tvt_slope_w{window}"] = None
            continue
        _, u_slope, u_scale = huber_line(known_md[-count:], visible_u[-count:], window_rows=count)
        _, tvt_slope, _ = huber_line(known_md[-count:], visible_tvt[-count:], window_rows=count)
        features[f"visible_u_slope_w{window}"] = u_slope
        features[f"visible_u_scale_w{window}"] = u_scale
        features[f"visible_u_slope_delta_w{window}"] = u_slope - full_u_slope
        features[f"visible_tvt_slope_w{window}"] = tvt_slope
    features.update(visible_backtests(known_md, known_z, visible_tvt, visible_backtest_fractions))
    features.update(read_typewell(typewell_path))
    truth = None
    if require_truth:
        truth = [float(value) for value in columns["TVT"] if value is not None]
    return {
        "features": features,
        "metadata": {
            "well_id": horizontal_path.name.split("__", 1)[0],
            "total_rows": total_rows,
            "known_rows": known_rows,
            "hidden_rows": hidden_rows,
            "last_visible_tvt": visible_tvt[-1],
            "typewell_sha256": hashlib.sha256(typewell_path.read_bytes()).hexdigest(),
        },
        "truth": truth,
    }


def predict_coefficients(model: Mapping[str, Any], features: Mapping[str, float | None]) -> tuple[float, float]:
    selected = list(model["selected_features"])
    medians = list(model["feature_medians"])
    means = list(model["feature_means"])
    scales = list(model["feature_scales"])
    coefficients = list(model["ridge_coefficients"])
    if not (len(selected) == len(medians) == len(means) == len(scales) == len(coefficients)):
        raise DeploymentDataError("model feature arrays have inconsistent lengths")
    row: list[float] = []
    for name, median_value, center, scale in zip(selected, medians, means, scales):
        raw = features.get(name)
        value = float(raw) if raw is not None and math.isfinite(float(raw)) else float(median_value)
        divisor = float(scale)
        if not math.isfinite(divisor) or abs(divisor) <= 1e-12:
            divisor = 1.0
        row.append((value - float(center)) / divisor)
    scaled = [
        sum(row[index] * float(coefficients[index][output]) for index in range(len(row)))
        for output in range(2)
    ]
    targets = [
        float(model["target_means"][output]) + float(model["target_scales"][output]) * scaled[output]
        for output in range(2)
    ]
    if not all(math.isfinite(value) for value in targets):
        raise DeploymentDataError("model produced non-finite coefficient")
    return targets[0], targets[1]


def reconstruct_hidden(last_visible_tvt: float, hidden_rows: int, datum: float, trend: float) -> list[float]:
    if hidden_rows <= 0:
        raise DeploymentDataError("hidden row count must be positive")
    predictions: list[float] = []
    for index in range(hidden_rows):
        centered = index / max(1, hidden_rows - 1) - 0.5 if hidden_rows > 1 else 0.0
        value = float(last_visible_tvt) + float(datum) + float(trend) * centered
        if not math.isfinite(value):
            raise DeploymentDataError("reconstructed non-finite prediction")
        predictions.append(value)
    return predictions


def load_model(path: Path) -> dict[str, Any]:
    model = json.loads(path.read_text(encoding="utf-8"))
    if model.get("schema_version") != 1 or model.get("experiment_id") != "E004":
        raise DeploymentDataError(f"{path}: unsupported E004 model schema")
    return model


def build_prediction_map(model: Mapping[str, Any], test_dir: Path) -> tuple[dict[str, float], list[dict[str, Any]]]:
    horizontal_files = sorted(test_dir.glob("*__horizontal_well.csv"))
    if not horizontal_files:
        raise DeploymentDataError(f"{test_dir}: no horizontal well files")
    predictions: dict[str, float] = {}
    well_rows: list[dict[str, Any]] = []
    for horizontal_path in horizontal_files:
        well_id = horizontal_path.name.split("__", 1)[0]
        typewell_path = test_dir / f"{well_id}__typewell.csv"
        if not typewell_path.exists():
            raise DeploymentDataError(f"missing typewell for {well_id}")
        extracted = extract_well_features(
            horizontal_path,
            typewell_path,
            visible_slope_windows=model["visible_slope_windows"],
            visible_backtest_fractions=model["visible_backtest_fractions"],
            require_truth=False,
        )
        datum, trend = predict_coefficients(model, extracted["features"])
        metadata = extracted["metadata"]
        values = reconstruct_hidden(metadata["last_visible_tvt"], metadata["hidden_rows"], datum, trend)
        for offset, value in enumerate(values, start=metadata["known_rows"]):
            key = f"{well_id}_{offset}"
            if key in predictions:
                raise DeploymentDataError(f"duplicate prediction id {key}")
            predictions[key] = value
        well_rows.append({
            "well_id": well_id,
            "known_rows": metadata["known_rows"],
            "hidden_rows": metadata["hidden_rows"],
            "predicted_datum": datum,
            "predicted_trend": trend,
        })
    return predictions, well_rows


def write_submission(model: Mapping[str, Any], test_dir: Path, sample_submission: Path, output_path: Path) -> dict[str, Any]:
    prediction_map, well_rows = build_prediction_map(model, test_dir)
    sample_ids: list[str] = []
    seen_ids: set[str] = set()
    with sample_submission.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        if list(reader.fieldnames or []) != ["id", "tvt"]:
            raise DeploymentDataError(f"{sample_submission}: expected columns id,tvt")
        for row in reader:
            key = str(row.get("id") or "")
            if not key:
                raise DeploymentDataError(f"{sample_submission}: empty id")
            if key in seen_ids:
                raise DeploymentDataError(f"{sample_submission}: duplicate id {key}")
            seen_ids.add(key)
            sample_ids.append(key)
    if set(sample_ids) != set(prediction_map):
        missing = sorted(set(sample_ids) - set(prediction_map))[:5]
        extra = sorted(set(prediction_map) - set(sample_ids))[:5]
        raise DeploymentDataError(f"sample/prediction ID mismatch; missing={missing}, extra={extra}")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "tvt"], lineterminator="\n")
        writer.writeheader()
        for key in sample_ids:
            writer.writerow({"id": key, "tvt": format(prediction_map[key], ".15g")})
    return {
        "rows": len(sample_ids),
        "wells": len(well_rows),
        "minimum_prediction": min(prediction_map.values()),
        "maximum_prediction": max(prediction_map.values()),
        "well_predictions": well_rows,
        "sha256": hashlib.sha256(output_path.read_bytes()).hexdigest(),
        "bytes": output_path.stat().st_size,
    }


In [ ]:
MODEL = json.loads(r'''{
  "code_sha": "c728a9a267a89c92e5c0c6b018db1a1658874158",
  "data_signature": "6ebe65b403f80fefd97dcd7bbfce7314252e779a1837c97364cf55761590fe77",
  "experiment_id": "E004",
  "external_artifacts_required": false,
  "feature_families": [
    "geometry",
    "prefix"
  ],
  "feature_maximums": [
    0.14693871362846955,
    0.0938484325634285,
    0.11450840756210062,
    0.09387154267521053,
    0.10974103321148976,
    0.23875139511816082,
    0.11170886826474619,
    241.14999999999964,
    0.07114664047474915,
    0.5580842562021723,
    0.5939925894894009,
    428.7600000000002,
    428.8100000000013,
    73.22969476993279,
    0.9588224418802304,
    90.92641460270225,
    124.0,
    7239.979999999981,
    123.23814728865699,
    60.29999999999927,
    0.8880050439748874,
    6558.689999999944,
    2666.509870588234,
    0.9999994143295278,
    2625.34,
    0.8751648351648351,
    63.90799999999974,
    8768.869999999995,
    0.17069379650882927,
    364.4800000000014,
    7536.850000000093,
    0.09189328949052705,
    156.73182954784542,
    389.12482931705705,
    10052.0,
    10052.0,
    10040.820186802426,
    0.07540997482077164,
    262.0500000000011,
    259.72426526615595,
    11565.168662432068,
    12140.0,
    12141.0,
    0.6558565123707739,
    1169.5699999999997,
    1185.0599999999995,
    428.7600000000002,
    126.21437803421382
  ],
  "feature_means": [
    -0.0006978742136164619,
    2.655398649136211e-05,
    1.623038977524215e-05,
    0.00021517593860865742,
    0.00033929245409915085,
    -0.002729837849293927,
    0.005270550621359208,
    11.225937904269124,
    0.005514549021218719,
    0.0017776957970893315,
    0.0017766510336394566,
    26.061228978007875,
    26.065213454075042,
    2.927045403550585,
    -0.06414874038533333,
    18.068165813587637,
    0.43141009055629015,
    -398.0449417852435,
    0.3773331920397501,
    -679.2476972833122,
    0.3930827002683297,
    -301.54548512289136,
    2098.1091724919825,
    0.10726970133120863,
    2103.124424320828,
    0.7331608207804723,
    0.26029596619633033,
    698.5842043984505,
    0.0056098430754583555,
    63.742509702457966,
    544.5713712807263,
    0.005485726559967859,
    6.9615075685812915,
    218.3469777929634,
    4895.199223803364,
    4895.199223803364,
    4887.058789604252,
    0.005297104607850565,
    2.775122897800788,
    2.4114850736621603,
    6123.08762067576,
    6586.652005174644,
    6587.652005174644,
    0.008317882398721582,
    716.5388486416556,
    722.3957826649415,
    165.3331306597671,
    47.91345529891506
  ],
  "feature_medians": [
    0.0006369264369884622,
    0.0004958990443744218,
    -0.00031586236413534696,
    0.0003556973297700744,
    0.0003444375492606816,
    -0.0005853160338561564,
    0.01189185633933434,
    18.93000000000029,
    0.024022168049461164,
    0.00024753058878285524,
    3.263356553568024e-05,
    100.02000000000044,
    100.03000000000065,
    2.015772569823742,
    -0.05285472820384153,
    16.307932439544977,
    0.11000000000240107,
    -297.79000000003725,
    0.04392736077478427,
    -660.6999999999989,
    0.40134431793124503,
    -232.89000000013039,
    2100.4353575757586,
    0.7264315719051245,
    2098.9900000000016,
    0.7399910700997172,
    0.12255102040861096,
    3570.0,
    0.01636869345038622,
    55.710000000000946,
    2500.909999999916,
    0.015897112795032985,
    4.440847002882389,
    225.9200235334002,
    4840.0,
    4840.0,
    4832.727291633304,
    0.016458325100165594,
    0.8000000000010914,
    0.5556610831028427,
    6085.1646725622795,
    6575.0,
    6576.0,
    0.0015082855980052248,
    740.1399999999994,
    747.7400000000016,
    160.5600000000013,
    46.17477243106832
  ],
  "feature_minimums": [
    -0.19651537914957168,
    -0.09605543133967534,
    -0.1330434131369005,
    -0.09599243833831113,
    -0.13102251615688787,
    -0.38701979359242267,
    -0.08201975759938475,
    -182.25,
    -0.056782320996082235,
    -0.10576612903225063,
    -0.1041761762551677,
    -343.5500000000011,
    -343.5799999999999,
    0.006386259760397372,
    -0.9976561389128178,
    3.312109429671559,
    -18.44999999999891,
    -7540.560000000056,
    -20.224079044117666,
    -1411.83,
    0.04747551598690518,
    -6574.939999999944,
    1606.5399540493963,
    -0.9999998427576628,
    1578.5900000000001,
    0.19776482021379982,
    -10.713333333333427,
    -10201.969999999972,
    -0.08455853116689876,
    12.460000000000946,
    -9141.589999999851,
    -0.08426384001365639,
    0.01001955036101264,
    28.657841123382415,
    407.0,
    407.0,
    406.2865595856577,
    -0.07653678657859489,
    -33.909999999999854,
    -39.217989448362474,
    1529.1179528732598,
    2057.0,
    2058.0,
    -0.08061832222572844,
    120.46000000000095,
    135.63000000000102,
    7.799999999999272,
    2.5857354314132714
  ],
  "feature_scales": [
    0.034635190072223584,
    0.02177287945940977,
    0.028147685379864874,
    0.02401967481367814,
    0.025297424193488106,
    0.05043100806932386,
    0.03529982269845044,
    56.78223048081566,
    0.034756186338669635,
    0.035491231161901386,
    0.037125320913169446,
    174.0339382045983,
    174.06708785291997,
    3.8970665607140074,
    0.5310811326660011,
    9.018539038516677,
    7.940313467357456,
    3314.428122585391,
    8.130586381324736,
    240.3755110691083,
    0.09054088160126093,
    2681.7115593880317,
    155.83756421423578,
    0.8380518962507546,
    152.73642904662825,
    0.0637343419618474,
    4.225138007925864,
    5247.341247166741,
    0.03644847144206917,
    32.92992359084372,
    4241.461055738747,
    0.035621509357576235,
    10.646229583338153,
    58.12869280176611,
    1300.3364950709376,
    1300.3364950709376,
    1298.3500375754688,
    0.03496762431833004,
    18.168764651002043,
    17.95717535857331,
    1293.9403385958017,
    1310.6149773574732,
    1310.6149773574732,
    0.04879527876052423,
    174.86115949435225,
    174.8498968139509,
    62.14421526870594,
    18.34433275593287
  ],
  "internet_required": false,
  "model_name": "geometry_prefix",
  "ridge_alpha": 25.0,
  "ridge_coefficients": [
    [
      -0.03916987115578119,
      -3.0445329988194035e-05,
      0.1170547377027878,
      0.01647568845629581
    ],
    [
      0.09146358147455287,
      0.02614307199678472,
      0.16020007922751953,
      0.1315837946508968
    ],
    [
      0.03324402852908378,
      -0.12158656871358811,
      0.07238108601708178,
      -0.016010614704689577
    ],
    [
      0.11841301039793754,
      0.018328773897025242,
      0.17515823106406558,
      -0.03253878883874896
    ],
    [
      -0.05230007267268964,
      0.05486303246042119,
      0.1965140096629869,
      0.006634690096643086
    ],
    [
      -0.17080587704867303,
      -0.0025715694807720967,
      -0.02080496450152159,
      -0.1957113842757028
    ],
    [
      -0.10931421673231063,
      -0.03303259540071277,
      -0.3960589249661517,
      0.0038495657728815336
    ],
    [
      0.06965053575358875,
      0.07252342092042854,
      -0.06853001600404834,
      -0.06514178647761727
    ],
    [
      0.06434405888885526,
      -0.46335591106118484,
      0.6135380648733032,
      0.08460596260823912
    ],
    [
      0.26831094815669854,
      -0.13378080536217865,
      -0.022101675974883016,
      0.18431715999919762
    ],
    [
      0.019641817348470307,
      0.10924391590442964,
      0.029194184363020917,
      -0.14198510717904425
    ],
    [
      -0.10692954692307374,
      -0.010153488884612462,
      0.196314977988444,
      0.035500946708954456
    ],
    [
      -0.10711315528115689,
      -0.010100568749078444,
      0.19623631222221014,
      0.03542880481749388
    ],
    [
      0.06791137420208643,
      0.07969111319847659,
      0.00874226828618619,
      0.2758828677733785
    ],
    [
      -0.06027900640659981,
      -0.1885213723299495,
      -0.2320361282988464,
      0.035103137423323126
    ],
    [
      0.017699036368403608,
      -0.0018604643690969262,
      -0.019349403622798144,
      -0.03526019730076477
    ],
    [
      -0.04244769558246789,
      -0.060509581075674855,
      -0.008686519325121243,
      -0.009780426962326325
    ],
    [
      -0.06298727626660557,
      0.03808672622525995,
      0.05325758868396682,
      0.02760873842665589
    ],
    [
      0.017695743638812367,
      0.012660037136708435,
      0.0012409197411071594,
      -0.02329704642736146
    ],
    [
      -0.05804112087601368,
      0.012163068047322633,
      0.08861634585127068,
      0.0212745152088931
    ],
    [
      -0.11209261694242047,
      -0.05617404831501182,
      -0.031192439041115384,
      -0.05726508538109242
    ],
    [
      -0.055011883171336734,
      0.04782237967811495,
      0.117347155120171,
      -0.007974034544598819
    ],
    [
      -0.008669511666299113,
      -0.0035413322458541893,
      0.03029117282153818,
      -0.014730066918115904
    ],
    [
      0.1655462818693918,
      0.2641454686654101,
      0.25620254952876437,
      0.009274769431743253
    ],
    [
      -0.03351409367972491,
      -0.00907400229172401,
      -0.0331766074805751,
      -0.019618781738110283
    ],
    [
      0.01947179282106812,
      0.060036621446025625,
      -0.001117952858132754,
      0.22759297789458918
    ],
    [
      -0.07140104850253211,
      -0.025667152732973323,
      -0.022620613721909533,
      -0.04271628633409607
    ],
    [
      0.03471308385262863,
      0.059899427972093626,
      -0.03226077689767145,
      -0.026249332587601946
    ],
    [
      -0.14216863939157928,
      0.0060866377930448865,
      -0.2471848944709616,
      0.008333121993840566
    ],
    [
      0.016806941171090294,
      -0.008351951304764505,
      0.02028582065850655,
      0.009342111861404047
    ],
    [
      0.01763183884761589,
      0.025335464362364467,
      -0.10822951713098206,
      -0.04222151380251153
    ],
    [
      -0.02848083879374127,
      -0.020375149320311405,
      -0.2743725983151259,
      -0.018126186934058198
    ],
    [
      0.004267197261800098,
      -0.04363363760716174,
      0.0052182170846246335,
      -0.012563495216981588
    ],
    [
      0.0030051343874174007,
      0.08217766643151181,
      -0.0044005741537651525,
      0.0723572749182411
    ],
    [
      0.022698061387727873,
      0.008262068997047166,
      0.001661844505844722,
      -0.02133403210355936
    ],
    [
      0.02269806138772788,
      0.008262068997047137,
      0.0016618445058447326,
      -0.021334032103559344
    ],
    [
      0.016717303762685813,
      0.00607208266144283,
      0.0010830505660217459,
      -0.035608748739180525
    ],
    [
      -0.05340216760794608,
      -0.017068211446006055,
      -0.30007165255248386,
      0.08581787145949576
    ],
    [
      0.017048417886220576,
      0.005568500423765481,
      -0.0017873253998620634,
      0.006735762515938269
    ],
    [
      -0.000650253061793892,
      0.0015353662228970763,
      -0.015098630710564824,
      -0.028923745613212974
    ],
    [
      -0.04600148397998339,
      -0.06252981390710001,
      -0.02004935827956715,
      0.014204092077083258
    ],
    [
      -0.009636308684406951,
      -0.028885393963791032,
      -0.007835376359780617,
      0.006145485233254057
    ],
    [
      -0.009636308684406951,
      -0.02888539396379103,
      -0.007835376359780618,
      0.006145485233254053
    ],
    [
      0.044316607370434614,
      0.06754095003968481,
      0.019407386837349223,
      0.023941215011373546
    ],
    [
      -0.004222196311581614,
      -0.003224493566927498,
      0.051273803964078817,
      -0.015130744453195882
    ],
    [
      0.010001067926926626,
      0.030250531084393493,
      0.04824991693807364,
      0.03194495161990494
    ],
    [
      -0.05091897666340323,
      0.023562370783328675,
      0.01784815488247395,
      0.13026373636257677
    ],
    [
      0.09806885767095062,
      -0.028537560673809494,
      -0.0047391446747100646,
      -0.11319850452739182
    ]
  ],
  "schema_version": 1,
  "selected_features": [
    "backtest_0p7_u_slope_delta",
    "visible_u_slope_delta_w512",
    "backtest_0p85_u_slope_delta",
    "visible_u_slope_delta_w128",
    "visible_u_slope_delta_w32",
    "backtest_0p5_u_slope_delta",
    "visible_u_slope",
    "visible_u_delta",
    "hidden_z_slope",
    "visible_tvt_slope_w32",
    "visible_tvt_slope_w128",
    "hidden_z_delta",
    "z_hidden_delta",
    "backtest_0p85_rmse",
    "hidden_azimuth_cos",
    "visible_u_std",
    "backtest_0p85_toe",
    "x_total_delta",
    "backtest_0p85_trend",
    "z_total_delta",
    "visible_tvt_slope",
    "x_hidden_delta",
    "visible_u_mean",
    "hidden_azimuth_sin",
    "last_visible_u",
    "hidden_fraction",
    "backtest_0p85_mean",
    "y_total_delta",
    "visible_u_slope_w32",
    "visible_u_range",
    "y_hidden_delta",
    "visible_u_slope_w128",
    "backtest_0p7_rmse",
    "visible_tvt_std",
    "hidden_rows",
    "md_hidden_span",
    "horizontal_hidden_distance",
    "visible_u_slope_w512",
    "backtest_0p7_toe",
    "backtest_0p7_trend",
    "horizontal_total_distance",
    "md_total_span",
    "total_rows",
    "visible_tvt_slope_w512",
    "visible_tvt_delta",
    "visible_tvt_range",
    "hidden_z_range",
    "hidden_z_std"
  ],
  "surfaces_required": false,
  "target_means": [
    1.42317883692573,
    1.0194283804439128,
    0.000489684642494126,
    2.4655967711641957
  ],
  "target_names": [
    "datum_correction_ft",
    "trend_correction_ft",
    "u_slope_delta_ft_per_md",
    "log1p_baseline_rmse"
  ],
  "target_scales": [
    12.690171253212025,
    21.210578247001994,
    0.022797200489427124,
    0.5547723172629699
  ],
  "training_wells": 773,
  "visible_backtest_fractions": [
    0.5,
    0.7,
    0.85
  ],
  "visible_slope_windows": [
    32,
    128,
    512
  ]
}
''')


In [ ]:
# Locate the competition input without assuming its Kaggle mount slug.
import os
from pathlib import Path

input_override = os.environ.get('ROGII_INPUT_ROOT', '').strip()
if input_override:
    competition_root = Path(input_override)
else:
    candidates = sorted(Path('/kaggle/input').glob('**/sample_submission.csv'))
    if not candidates:
        raise FileNotFoundError('Could not locate sample_submission.csv under /kaggle/input')
    competition_root = candidates[0].parent

test_dir = competition_root / 'test'
sample_submission = competition_root / 'sample_submission.csv'
output_path = Path(os.environ.get('ROGII_OUTPUT_PATH', '/kaggle/working/submission.csv'))
result = write_submission(MODEL, test_dir, sample_submission, output_path)
print(json.dumps({k: v for k, v in result.items() if k != 'well_predictions'}, indent=2, sort_keys=True))
